# Model Gateway, Routing, Fallback (from scratch)

Build a **task router** and a **fallback chain** you can actually run. A mock LLM
stands in **offline**, so no keys are needed; the real LiteLLM path is shown and
runs when `LIVE` is True. Includes the honest note on when a single-provider
Bedrock stack does **not** need this.

Pair with the deck's Segment 3 and `release_pipeline.md` (the fallback path ties
back to the eval gate).

## Setup

**VS Code (3 steps)**
```bash
python -m venv .venv && source .venv/bin/activate     # 1. activate venv
aws configure                                         # 2. creds (only needed if LIVE)
pip install litellm boto3                              # 3. install
```

**Colab (3 steps)**
```python
!pip install -q litellm boto3                          # 1. install
import os                                               # 2. creds via Colab Secrets (if LIVE)
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"         # 3. region
```

**LIVE flag:** offline by default. The mock LLM makes every cell run with no
network. Flip `LIVE = True` to use real LiteLLM against Bedrock.

In [1]:
# ---- config + a mock LLM so this runs offline ----
LIVE = False

# task name -> Bedrock model id (the real router maps names to these)
MODELS = {
    "plan":      "bedrock/us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # strong, reasoning
    "summarize": "bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0",   # cheap, easy work
}

def mock_llm(model, prompt, fail=False):
    """Deterministic stand-in for a model call. No network.

    fail=True simulates a 429 throttle so we can demonstrate fallback.
    """
    if fail:
        raise RuntimeError("429 throttled")
    tag = "sonnet" if "sonnet" in model else "haiku"
    return f"[{tag}] handled: {prompt[:42]}"

print("offline mock ready. MODELS:", list(MODELS))

offline mock ready. MODELS: ['plan', 'summarize']


## Step 1. The problem: a hardcoded model

Calling one fixed model for everything is wasteful (the strong model on an easy
task) and fragile (one provider, no failover).

In [2]:
def naive(prompt):
    return mock_llm(MODELS["plan"], prompt)   # always the strong, expensive model

print(naive("Shorten this itinerary note"))   # overkill for an easy task

[sonnet] handled: Shorten this itinerary note


## Step 2. Task-based routing

Send each task to the right model by **name**, not by hardcoding an id. Planning
goes to the strong model, summarizing to the cheap one.

In [3]:
def route(task, prompt, fail=False):
    """Resolve a task name to a model and call it."""
    return mock_llm(MODELS[task], prompt, fail=fail)

print(route("plan",      "Re-plan JX48Q2 around the storm"))
print(route("summarize", "Shorten this itinerary note"))

[sonnet] handled: Re-plan JX48Q2 around the storm
[haiku] handled: Shorten this itinerary note


In [4]:
# The same thing with real LiteLLM (runs only when LIVE). The Router gives you
# one endpoint, names instead of ids, plus built-in retries, timeouts, cooldowns.
if LIVE:
    from litellm import Router
    router = Router(
        model_list=[
            {"model_name": "plan",      "litellm_params": {"model": MODELS["plan"]}},
            {"model_name": "summarize", "litellm_params": {"model": MODELS["summarize"]}},
        ],
        fallbacks=[{"plan": ["summarize"]}],   # if plan fails, try summarize
    )
    r = router.completion(model="plan", messages=[{"role": "user", "content": "Re-plan JX48Q2"}])
    print(r.choices[0].message.content)
else:
    print("offline: using the mock router above. Set LIVE=True for real LiteLLM + Bedrock.")

offline: using the mock router above. Set LIVE=True for real LiteLLM + Bedrock.


## Step 3. A fallback chain

Order the chain by cost: primary, then a cheaper or same-provider backup. If the
primary fails, move to the next link. Caller code does not change.

In [5]:
def with_fallback(chain, prompt, break_primary=False):
    """Try each task in order; return the first success."""
    last = None
    for i, task in enumerate(chain):
        try:
            fail = break_primary and i == 0     # simulate the primary throttling
            out = route(task, prompt, fail=fail)
            print(f"  [{task}] ok")
            return out
        except Exception as e:
            print(f"  [{task}] {e} -> falling over")
            last = e
    raise last

print("normal run:")
with_fallback(["plan", "summarize"], "Re-plan JX48Q2")

print("\nforced primary failure:")
with_fallback(["plan", "summarize"], "Re-plan JX48Q2", break_primary=True)

normal run:
  [plan] ok

forced primary failure:
  [plan] 429 throttled -> falling over
  [summarize] ok


'[haiku] handled: Re-plan JX48Q2'

## Step 4. Bound the fallback with a circuit breaker

A naive fallback retries an unhealthy primary on **every** request and amplifies
the outage. A circuit breaker trips after a few failures, skips the bad target
during a cooldown, then probes once before trusting it again
(closed -> open -> half-open). Community defaults: about 5 failures, 60s cooldown.

In [6]:
import time

class CircuitBreaker:
    def __init__(self, threshold=2, cooldown=1.0):
        self.threshold = threshold      # fails before the circuit opens
        self.cooldown = cooldown        # seconds to skip the target
        self.fails = 0
        self.open_until = 0.0

    def call(self, fn, *args, **kwargs):
        if time.time() < self.open_until:
            raise RuntimeError("circuit OPEN: skipping unhealthy target")
        try:
            out = fn(*args, **kwargs)
            self.fails = 0              # success closes the circuit
            return out
        except Exception:
            self.fails += 1
            if self.fails >= self.threshold:
                self.open_until = time.time() + self.cooldown
                print(f"  circuit OPEN for {self.cooldown}s after {self.fails} failures")
            raise

cb = CircuitBreaker(threshold=2, cooldown=1.0)
for i in range(4):
    try:
        cb.call(route, "plan", "x", fail=True)   # primary keeps failing
    except Exception as e:
        print(f"  attempt {i+1}: {e}")

  attempt 1: 429 throttled
  circuit OPEN for 1.0s after 2 failures
  attempt 2: 429 throttled
  attempt 3: circuit OPEN: skipping unhealthy target
  attempt 4: circuit OPEN: skipping unhealthy target


## Step 5. The honest alternative: you may not need a gateway

On a **single-provider Bedrock stack**, Strands already abstracts the model and
Bedrock handles cross-region routing. The model swap is one line, no third party:

```python
m = BedrockModel(model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0")  # strong
m = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")   # cheap
```

Reach for LiteLLM when you actually have more than one provider (Bedrock +
OpenAI + local), or want per-team budgets, virtual keys, and one cost dashboard
across providers. Do not bolt a gateway onto a single-provider app for its own
sake.

## Production notes

| Practice | Why |
|---|---|
| Alert when the secondary handles more than ~5% of traffic | a fallback that fires often is a hidden outage, not resilience |
| Pin model versions | an auto-update can change the prompt format and silently break routing |
| Size timeouts for cold start | a slow first call can trip a false fallback |
| Evaluate the fallback path, not just the primary | a backup can answer while quality drops; the dashboard stays green |
| Review the chain quarterly | models, prices, and rate limits change |

The last row connects back to release engineering: routing resilience and the
eval gate are one problem, not two.